# 15 — Data Contracts

### Why this exists
A contract is useful only if it changes what crosses a boundary. This lab makes a producer publish versioned payloads, validates them at the consumer boundary, and makes compatibility a measured behavior.

### Learning objectives
model a contract; validate good/bad payloads; distinguish compatible from breaking evolution; quarantine invalid records; produce a contract report

### Prerequisite
Python dictionaries and basic exception handling

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/15-data-contracts.svg)

> **Question to carry through the lab:** What happens when a producer changes a field while an older consumer is still running?

## Capability contract

**Capability:** C03 — Correctness Contracts & Idempotency  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Detect a correctness failure


## Mental model
Treat a contract as executable boundary policy: schema + compatibility rules + ownership + failure handling. Version numbers alone do not make an evolution safe.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S02 — API schema drift (DATA-1051)** → [`../work_simulations/02_acme_commerce_api_schema_drift/README.md`](../work_simulations/02_acme_commerce_api_schema_drift/README.md)
- **S03 — Data quality (DATA-1063)** → [`../work_simulations/03_acme_commerce_quality_gate/README.md`](../work_simulations/03_acme_commerce_quality_gate/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass(frozen=True)
class Contract:
    version: int
    required: dict[str, type]
    optional: dict[str, type]

    def validate(self, payload: dict[str, Any]):
        errors=[]
        for k,t in self.required.items():
            if k not in payload: errors.append(f"missing:{k}")
            elif not isinstance(payload[k],t): errors.append(f"type:{k}")
        for k,t in self.optional.items():
            if k in payload and not isinstance(payload[k],t): errors.append(f"type:{k}")
        return errors

v1=Contract(1,{"order_id":str,"amount":float},{"coupon":str})
good={"order_id":"o-1","amount":120.0,"coupon":"SAVE10"}
print(v1.validate(good))

In [ ]:
payloads=[
    ("good",good),
    ("missing_amount",{"order_id":"o-2"}),
    ("wrong_type",{"order_id":"o-3","amount":"120"}),
]
for name,p in payloads:
    print(name, v1.validate(p))

In [ ]:
# Compatibility is a consumer question: can every payload accepted before still be read?
def compatible(old: Contract, new: Contract):
    breaking=[]
    for field,t in old.required.items():
        if field not in new.required and field not in new.optional: breaking.append(f"removed:{field}")
        elif field in new.required and new.required[field] is not t: breaking.append(f"changed:{field}")
    for field,t in new.required.items():
        if field not in old.required and field not in old.optional: breaking.append(f"new_required:{field}")
    return breaking

v2_add_optional=Contract(2,{"order_id":str,"amount":float},{"coupon":str,"currency":str})
v2_breaking=Contract(2,{"order_id":str,"amount":float,"currency":str},{"coupon":str})
print("optional evolution",compatible(v1,v2_add_optional))
print("breaking evolution",compatible(v1,v2_breaking))

In [ ]:
# Consumer boundary: accepted records continue; invalid records are quarantined.
def ingest(records, contract):
    accepted=[]; quarantine=[]
    for r in records:
        (accepted if not contract.validate(r) else quarantine).append(r)
    return accepted, quarantine
accepted, quarantine=ingest([p for _,p in payloads],v1)
report={"accepted":len(accepted),"quarantined":len(quarantine),"quarantine_reasons":[v1.validate(r) for r in quarantine]}
print(report)

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
In production, contract ownership, compatibility policy, rollout order, quarantine/replay and observability matter as much as the schema itself.

## 9. Independent challenge
Make a third evolution: rename amount to order_amount. Decide whether to accept an alias, publish v3, or reject it. Measure how many historical payloads remain readable.

### Evidence to keep
contract definitions, compatibility report, accepted/quarantined counts, and the decision note

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.